<a href="https://colab.research.google.com/github/kasrasa/Object-detection-tutorial/blob/YOLO/YOLO_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U pycocotools
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.0 MB/s eta 0:00:00


In [2]:
import os
import random
import shutil
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pycocotools.coco import COCO
from IPython.display import display

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch version: 2.11.0+cu128
CUDA available: True


In [3]:
# -------------------------
# Global configuration
# -------------------------

SEED = 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # Ultralytics accepts GPU index or "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# COCO annotation paths uploaded in Colab.
TRAIN_ANN = "/content/data/train/instances_train2014.json"
VAL_ANN = "/content/data/valid/instances_val2014.json"
TRAIN_INSTANCES_ROOT = Path("/content/data/train/")
VAL_INSTANCES_ROOT = Path("/content/data/valid/")
TRAIN_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)
VAL_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)

# Image roots. Images are downloaded on demand into these folders.
IMAGE_ROOT = Path("/content/data/images")
LABEL_ROOT = Path("/content/data/labels")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
LABEL_ROOT.mkdir(parents=True, exist_ok=True)

# YOLO-native exported dataset root.
YOLO_DATA_ROOT = Path("/content/data/")
YOLO_ORIGINAL_ROOT = YOLO_DATA_ROOT / "original"
YOLO_EXPANDED_ROOT = YOLO_DATA_ROOT / "expanded"

# Dataset sizes.
NUM_TRAIN = 200
NUM_VAL = 50
MIN_SMALL_OBJECTS_PER_IMAGE = 3
NUM_ADDED_HARD_IMAGES = 100

# YOLO model and training settings.
# Change to "yolo11n.pt" or "yolov8n.pt" if you want an older baseline.
YOLO_WEIGHTS = "yolo26n.pt"
IMG_SIZE = 640
BATCH_SIZE = 8
NUM_WORKERS = 4
NUM_EPOCHS = 10
PATIENCE = 0
FREEZE_LAYERS = None  # Example: 10 to freeze early layers. None means no freeze argument.

# Evaluation / mining settings.
IOU_THRESH = 0.5 # regular iou threshold to match detected bbs to ground truth
SCORE_THRESH = 0.05 # intentionally low to see if model can detect objects with low confidence or completely misses them
POOR_RECALL_THRESHOLD = 0.7 # used to find weak classes and candidate classes for hard mining
MIN_SMALL_GT = 3 # number of small gt objects in the image that is not ocluded or crowded

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Using Ultralytics device:", DEVICE)
print("Train annotations exist:", os.path.exists(TRAIN_ANN))
print("Val annotations exist:", os.path.exists(VAL_ANN))

Using Ultralytics device: 0
Train annotations exist: False
Val annotations exist: False


In [7]:
coco_train = COCO(TRAIN_ANN)
coco_val = COCO(VAL_ANN)

loading annotations into memory...
Done (t=8.21s)
creating index...
index created!
loading annotations into memory...
Done (t=4.59s)
creating index...
index created!


In [8]:
def build_coco_yolo_category_maps(coco):
  coco_to_yolo = {}
  yolo_to_coco = {}
  class_names = []
  cat_ids = sorted(coco.getCatIds())
  categories = coco.loadCats(cat_ids)
  for idx, cat in enumerate(categories):
    coco_to_yolo[cat["id"]] = idx # coco ids mapped to yolo
    yolo_to_coco[idx] = cat["id"] # yolo ids mapped to coco
    class_names.append(cat["name"])
  return coco_to_yolo, yolo_to_coco, class_names

def validate_ann(ann):
  if ann["iscrowd"] == 1:
    return False

  x, y, w, h = ann["bbox"]
  if ann.get("area", w*h) <= 1:
    return False
  if ann["bbox"][2] <= 1 or ann["bbox"][3] <= 1:
    return False
  return True

def coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo):
    x, y, w, h = ann["bbox"]

    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    bw = w / img_w
    bh = h / img_h

    class_id = coco_to_yolo[ann["category_id"]]

    return [class_id, cx, cy, bw, bh]

def create_yolo_dataset(anns, coco, coco_to_yolo):
  yolo_dataset = defaultdict(list)

  for ann in anns:
    image_id = ann["image_id"]
    image_info = coco.loadImgs(image_id)[0]

    img_w, img_h = image_info["width"], image_info["height"]
    yolo_row = coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo)
    yolo_dataset[image_id].append(yolo_row)
  return yolo_dataset

coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_train)
anns = coco_train.loadAnns(coco_train.getAnnIds())
yolo_dataset_train = create_yolo_dataset(anns, coco_train, coco_to_yolo)
coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_val)
anns = coco_val.loadAnns(coco_val.getAnnIds())
yolo_dataset_val = create_yolo_dataset(anns, coco_val, coco_to_yolo)
print("train images:", len(yolo_dataset_train))
print("val images:", len(yolo_dataset_val))

train images: 82081
val images: 40137


In [9]:
def classify_bb_area(anns):
  buckets = {
      "small": defaultdict(list),
      "medium": defaultdict(list),
      "large": defaultdict(list),
      "all": defaultdict(list)
  }

  for ann in anns:
    if not validate_ann(ann):
      continue

    image_id = ann["image_id"]
    area = ann["area"]

    if area < 32*32:
      buckets["small"][image_id].append(ann)
    elif area < 96*96:
      buckets["medium"][image_id].append(ann)
    else:
      buckets["large"][image_id].append(ann)

    buckets["all"][image_id].append(ann)
  return buckets

def get_images_by_object_size(
    coco,
    size_bucket,
    category_ids=None,
    min_objects=MIN_SMALL_OBJECTS_PER_IMAGE,
):
    selected_image_ids = []

    image_ids = coco.getImgIds()
    ann_ids = coco.getAnnIds(imgIds=image_ids, iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    buckets = classify_bb_area(anns)

    for image_id, bucket_anns in buckets[size_bucket].items():
      if len(bucket_anns) >= min_objects:
        for ann in bucket_anns:
          if category_ids is not None and ann["category_id"] not in category_ids:
            continue

          selected_image_ids.append(image_id)

    return selected_image_ids

def train_val_images(selected_image_ids_train, selected_image_ids_val, num_train, num_val, seed = SEED):
  random.seed(seed)

  train_image_ids = []
  val_image_ids = []

  train_image_ids = random.sample(selected_image_ids_train, min(num_train, len(selected_image_ids_train)))
  val_image_ids = random.sample(selected_image_ids_val, min(num_val, len(selected_image_ids_val)))

  return train_image_ids, val_image_ids

selected_image_ids = get_images_by_object_size(coco_train, "all")
print("selected images:", len(selected_image_ids))
print(selected_image_ids[:10])
yolo_dataset_train[selected_image_ids[0]]

selected_image_ids_train = get_images_by_object_size(coco_train, "all", min_objects=MIN_SMALL_OBJECTS_PER_IMAGE)
selected_image_ids_val = get_images_by_object_size(coco_val, "all", min_objects=MIN_SMALL_OBJECTS_PER_IMAGE)
selected_image_ids_train, selected_image_ids_val = train_val_images(
    selected_image_ids_train, selected_image_ids_val,
    NUM_TRAIN, NUM_VAL
)
print("train images:", len(selected_image_ids_train))
print("val images:", len(selected_image_ids_val))

selected images: 558503
[57870, 57870, 57870, 57870, 57870, 57870, 57870, 57870, 57870, 57870]
train images: 200
val images: 50


In [10]:
def create_yolo_txt_files(label_path, coco, image_ids, yolo_dataset, split = "train"):
  dataset_root = Path(label_path)

  label_dir = dataset_root / split

  label_dir.mkdir(parents=True, exist_ok=True)

  for image_id in image_ids:
    image_info = coco.loadImgs(image_id)[0]
    file_name = image_info["file_name"]

    label_file = label_dir / file_name.replace(".jpg", ".txt")

    yolo_rows = yolo_dataset.get(image_id,[])
    if len(yolo_rows) == 0:
      print(f"Warning: no labels for image_id={image_id}, file={file_name}")

    with open(label_file, "w", encoding="utf-8") as f:
      for row in yolo_rows:
        class_id, cx, cy, bw, bh = row
        f.write(
                    f"{int(class_id)} "
                    f"{cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"
                )

def download_image_file(image_path, coco, image_ids, split = "train"):
  dataset_root = Path(image_path)

  image_dir = dataset_root / split

  image_dir.mkdir(parents=True, exist_ok=True)

  for image_id in image_ids:
    image_info = coco.loadImgs(image_id)[0]
    file_name = image_info["file_name"]

    image_file = image_dir / file_name

    if not image_file.exists():
      image_url = f"http://images.cocodataset.org/{split}2014/{file_name}"
      urllib.request.urlretrieve(image_url, image_file)


def write_yolo_yaml(dataset_path, class_names):
  root = Path(dataset_path)

  with open(root/"data.yaml", "w", encoding="utf-8") as f:
    f.write(
        f"path: {dataset_path}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"nc: {len(class_names)}\n"
        f"names: {class_names}\n"
    )

create_yolo_txt_files(LABEL_ROOT, coco_train, selected_image_ids_train, yolo_dataset_train)
create_yolo_txt_files(LABEL_ROOT, coco_val, selected_image_ids_val, yolo_dataset_val, "val")
download_image_file(IMAGE_ROOT, coco_train, selected_image_ids_train)
download_image_file(IMAGE_ROOT, coco_val, selected_image_ids_val, "val")
_, _, class_names = build_coco_yolo_category_maps(coco_train)
write_yolo_yaml(YOLO_DATA_ROOT, class_names)
label_files = list((Path(LABEL_ROOT) / "train").glob("*.txt"))
print("label files:", len(label_files))

first_label = label_files[0]
print("first label file:", first_label)

with open(first_label, "r") as f:
    print(f.read())

label files: 199
first label file: /content/data/labels/train/COCO_train2014_000000069486.txt
56 0.747148 0.936594 0.223547 0.125562
56 0.897344 0.952906 0.156875 0.055688
56 0.977648 0.955187 0.044516 0.058083
0 0.161797 0.832583 0.249437 0.312375
0 0.446820 0.631344 0.135734 0.136146
0 0.549094 0.758771 0.125688 0.218708
46 0.212047 0.277292 0.077406 0.120625
46 0.644125 0.231365 0.060000 0.087063
46 0.542570 0.241958 0.046359 0.112042
46 0.837016 0.236792 0.089375 0.126208
46 0.397227 0.258573 0.025297 0.064979
46 0.635703 0.233542 0.071250 0.114583
46 0.307531 0.266333 0.046500 0.103583
46 0.461086 0.255740 0.103953 0.111604
46 0.757922 0.252635 0.025625 0.060396
46 0.955398 0.223052 0.072922 0.151312
47 0.810055 0.593042 0.110078 0.120917
49 0.581461 0.334833 0.755047 0.193250
49 0.654430 0.488406 0.020484 0.031146
49 0.696914 0.476646 0.020016 0.025000
49 0.778773 0.450177 0.023234 0.030313
56 0.077352 0.927021 0.094391 0.145208
56 0.396477 0.661594 0.043734 0.055771
56 0.177992 

In [11]:
model = YOLO(YOLO_WEIGHTS)
outputs = model.train(data = "data/data.yaml", epochs = NUM_EPOCHS, device=-1)

Searching for 1 idle GPUs with free memory >= 20.0% and free utilization >= 0.0%...
Selected idle CUDA devices [0]
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

AttributeError: 'DetMetrics' object has no attribute 'shape'. See valid attributes below.
Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

    Attributes:
        names (dict[int, str]): A dictionary of class names.
        box (Metric): An instance of the Metric class for storing detection results.
        speed (dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
        stats (dict[str, list]): A dictionary containing lists for true positives, confidence scores, predicted classes,
            target classes, and target images.
        nt_per_class: Number of targets per class.
        nt_per_image: Number of targets per image.

    Methods:
        update_stats: Update statistics by appending new values to existing stat collections.
        process: Process predicted results for object detection and update metrics.
        clear_stats: Clear the stored statistics.
        

In [14]:
print(f"stats: {outputs.stats}")
print(f"box: {outputs.box}")
print(f"instances per class: {outputs.nt_per_class}")
print(f"instances per image: {outputs.nt_per_image}")

stats: {'tp': [], 'conf': [], 'pred_cls': [], 'target_cls': [], 'target_img': []}
box: ultralytics.utils.metrics.Metric object with attributes:

all_ap: array([[    0.73002,     0.69505,     0.64098,     0.61753,     0.54629,     0.46617,     0.40085,     0.30572,     0.17364,    0.036159],
       [    0.48621,     0.43689,     0.42584,     0.35532,     0.26365,     0.20719,     0.16998,     0.15974,     0.15974,   0.0046774],
       [     0.5394,     0.51161,     0.45801,      0.4274,     0.37481,     0.34879,     0.26171,     0.17865,      0.0848,    0.012044],
       [    0.59885,     0.59885,     0.58948,     0.58542,     0.57042,     0.42353,     0.32526,     0.26714,         0.1,           0],
       [      0.665,       0.665,       0.665,       0.665,       0.665,       0.665,       0.665,       0.665,       0.335,           0],
       [        0.5,         0.5,         0.5,         0.5,         0.5,         0.5,         0.5,         0.5,       0.285,       0.285],
       [    0